# Word Embeddings (Word2Vec)

## 1. What Are Word Embeddings?

- Word embeddings are dense vector representations of words that capture their semantic meaning and similarity.
- Words with similar meanings or context have similar vector values.
- Unlike one-hot encoding or TF-IDF (which are sparse and context-blind), Word2Vec learns representations based on how words appear together in sentences.

Representation comparison (example for the word "bank"):
- One-hot Encoding: [0, 0, 0, 1, 0, 0, ..., 0] -> Very high-dimensional and sparse.
- TF-IDF: [0.0, 0.8, 0.1, 0.0, ...] -> High-dimensional and sparse.
- Word2Vec (Embedding): [0.21, -0.43, 0.65, ...] -> Low-dimensional and dense.

## 2. How Word2Vec Works

- Word2Vec uses a neural network to learn relationships between words based on their context (neighboring words).
- It has two main architectures:
  - CBOW (Continuous Bag of Words)
  - Skip-gram

## 3. Training Data and Learning

- Trained on a large text dataset containing documents.
  - Examples: news articles, Wikipedia, etc.
- It learns by analyzing:
  - Which words appear together.
  - Which words appear in similar contexts.
- Main idea: Words appearing in similar contexts get similar vectors.

## 4. CBOW (Continuous Bag of Words)

- CBOW's job is to predict the missing center word from surrounding context words.

Example 1:
- Sentence: "Customer salary ___ due to bank holiday."
- Context words: [Customer, salary, due, to, bank, holiday]
- Model predicts: delayed
- Learning: Since words like "salary", "bank", and "holiday" frequently appear near "delayed", Word2Vec learns their relationship.

Example 2:
- Sentence: "FD amount will be automatically ___."
- Context words: [FD, amount, will, be, automatically]
- Model predicts: renewed

Note: CBOW is not answering a question; it is just predicting which word fits best in that position based on surrounding words.

Why CBOW cannot handle factual QA (example: "Tell me FD interest for 1 year." -> 8%):

What happens:
- It is not predicting a missing word.
- It needs a real answer (8%) from external knowledge (e.g. an FD rate table).
- It requires understanding the query meaning, retrieving the correct rate from data/memory, and responding.

Why CBOW cannot do this:
- CBOW does not store facts like "FD = 8%".
- It only learns statistical word patterns, such as:
  - "FD" often appears near "interest".
  - "interest" often appears near "rate".
  - "1 year" often appears near "tenure".
- It learns relationships like: similar(FD, deposit), similar(interest, rate).
- It cannot look up or calculate "interest rate = 8%" - that needs retrieval or QA models (e.g. BERT QA, ChatGPT, etc.)

## 5. Skip-gram (Predict Context Words From Center Word)

- Skip-gram is the reverse of CBOW. Instead of predicting the center word from its context, it predicts the context words from the center word.
- CBOW: Context -> predicts Center word.
- Skip-gram: Center word -> predicts Context words.

Why Skip-gram was introduced:
- CBOW works well for frequent words, but Skip-gram works better for rare words, because one center word can generate many (context, target) pairs.
- This helps the model learn word relationships even when some words appear less often.

Example 1:
- Target word: EMI
- Model predicts nearby words: loan, payment, due, monthly
- Learning: Since these words frequently appear around "EMI" in customer emails/messages, Word2Vec learns they are semantically related.

Example 2:
- Target word: autorenewal
- Model predicts nearby words: FD, maturity, renewed
- Learning: These co-occurrences help Word2Vec learn their relationship.

Step-by-step Skip-gram example:
- Sentence: "Senior citizens get higher interest on FD."
- Center word: "FD"
- Context window size: 2 (i.e. 2 words before and 2 after)
- Input (center word): "FD"
- Output (context words): [interest, on]
- The model learns: when "FD" appears, nearby words like "interest", "deposit", "maturity", and "rate" are very likely.

What the model actually learns:
Through millions of sentences, the model sees patterns like:
- Sentence "FD interest rate is high." with center word "FD" -> likely contexts: [interest, rate]
- Sentence "FD maturity period is 2 years." with center word "FD" -> likely contexts: [maturity, period, years]
- Sentence "Senior citizen FD has higher return." with center word "FD" -> likely contexts: [higher, return]

So the model learns semantic closeness:
- "FD" vector becomes close to "deposit", "investment", "maturity", etc.
- "interest" vector becomes close to "rate", "return", etc.
- This is how Skip-gram helps build a meaning network among words.

## 6. Average Word2Vec (Sentence / Document Representation)

What is Average Word2Vec:
- A simple way to represent a sentence or document by taking the average of all its word embeddings.
- Each word has a vector (from CBOW or Skip-gram training). If we take the average of these word vectors, we get a single dense vector that represents the overall meaning of the sentence.

Why we need it:
- Machine learning models (like classifiers) need fixed-length inputs, but sentences have variable lengths (5 words, 20 words, etc.).
- To solve this, we take the average embedding of all words in a sentence. That gives us a single fixed-size vector even if the sentence has many words.

Why it helps (semantic similarity):
- Sentence 1: "What is the FD rate for 2 years?"
- Sentence 2: "Tell me the return for a 24-month deposit."
- Even though the words are different, "FD" is close to "deposit" and "rate" is close to "return", so the average vectors become similar because Word2Vec captures semantic meaning.

FD use case - customer query classifier:
- Query 1: "What is the FD rate for 2 years?"
- Query 2: "Tell me the return for a 24-month deposit."
- Average Word2Vec converts both sentences into similar vectors, helping models like Logistic Regression classify both into the same intent: Rate Enquiry.

How it works, step by step:
- Sentence: "What is the interest rate for a 2-year FD?"
- Example word embeddings (3-dimensional vectors):
  - interest: [0.65, 0.12, -0.45]
  - rate: [0.62, 0.09, -0.49]
  - FD: [0.70, 0.15, -0.40]
  - What: [0.10, -0.20, 0.30]
  - is: [0.05, -0.10, 0.12]
  - the: [0.02, -0.05, 0.10]
  - for: [0.04, -0.08, 0.11]
  - a: [0.01, -0.02, 0.03]
  - 2-year: [0.30, 0.05, -0.20]
- Process:
  - Step 1: Add all word vectors (element-wise sum).
  - Step 2: Divide by total number of words (9 here).
  - Step 3: Final output -> one dense vector (3-dimensional).
- Final average vector (example): [0.28, -0.00, -0.10]
  - This represents the overall meaning of the sentence.

Advantages:
- Converts variable-length text into fixed-size numeric vectors.
- Simple and fast to compute.
- Captures overall meaning of short sentences effectively.

Limitations:
- Loses word order: "Sandipan report to Anand" and "Anand report to Sandipan" will have the same average vector.
- Treats all words equally: important words ("interest", "FD") and unimportant words ("is", "for") get equal weight.
- Context-insensitive: the same word vector is used in every situation (no difference between "bank account" and "river bank").

## Key Takeaways

- Word2Vec learns word relationships from context (using CBOW or Skip-gram).
- CBOW: Context -> Center word (best for frequent words).
- Skip-gram: Center word -> Context words (better for rare words).
- Average Word2Vec turns any sentence into a fixed-size vector for downstream tasks.
- Great for similarity, clustering, classification, and NLP applications.